# 3차 실험 모델 선정
2차 실험 run의 클래스별 AP에 train 객체 수를 붙여, train 객체가 적은 클래스(희소 절반)와 많은 클래스(빈번 절반)의 AP 차이를 모델끼리 비교합니다. 3차 실험(오버/언더샘플링, 클래스별 손실 가중치)에 쓸 모델을 고르는 근거로 씁니다.

**쓰는 법**
1. **각자 바꿀 값** 셀의 `RUNS`에 비교할 2차 run 폴더와 그 run의 방법(v2, v3, v4)을 직접 적습니다. 폴더에는 채점 때 저장한 `scores.json`이 있어야 합니다.
2. 위에서부터 실행합니다. 학습을 하지 않으므로 GPU 런타임이 필요 없습니다.

**판단 기준**
- 같은 방법끼리 비교합니다. 방법마다 train 객체 수가 달라서, 방법이 다른 run의 gap은 비교하지 않습니다.
- ① gap(빈번 절반 AP − 희소 절반 AP)이 큰 모델: 불균형의 영향을 크게 받아 3차에서 개선할 여지가 큽니다.
- ② val 여유(1 − val)가 있는 모델: val이 1에 가까우면 3차의 개선이 val에 드러나지 않습니다.
- RT-DETR는 클래스별 손실 가중치(`cls_pw`)가 손실에 반영되지 않아, 고르면 샘플링 조건만 돌립니다. (`src/pill_detection/balance.py` 모듈 docstring)

In [ ]:
# 각자 바꿀 값. 값을 바꾸면 이 셀부터 다시 실행한다
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/pill data')   # pill data 바로가기를 다른 곳에 뒀으면 그 경로
BRANCH = 'feat/41-exp3-balance'                        # balance.py가 있는 브랜치. main에 병합되면 'main'으로 바꾼다
RUNS = {   # run 이름: (run 폴더, 그 run을 학습한 방법). scores.json이 있는 폴더를 직접 적는다
    'v3_frcnn_e20_s0': (DRIVE_DIR / 'rw_runs/v3_frcnn_e20_s0', 'v3'),
    # 'v3_yolo26s': (DRIVE_DIR / '<이름>_runs/v3_yolo26s', 'v3'),
    # 'v3_rtdetr-l': (DRIVE_DIR / '<이름>_runs/v3_rtdetr-l', 'v3'),
}
N_BOTTOM = 5   # train 객체가 가장 적은 클래스를 몇 개 보여줄지

In [ ]:
# 세션 시작 (Drive 연결, 레포 받기). 학습을 하지 않으므로 패키지 설치와 W&B 로그인은 하지 않는다
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/pill-object-detection'
!test -d {REPO} || git clone -q https://github.com/codeit-ai-15th/pill-object-detection.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git switch -q -C {BRANCH} FETCH_HEAD && git log -1 --format='%h %s'

import sys; sys.path.insert(0, f'{REPO}/src')

In [ ]:
# 모델 선정 표: run마다 val 점수와 희소/빈번 절반의 AP, 그리고 train 객체가 가장 적은 클래스
import json

from pill_detection.balance import class_ap_rows, imbalance_summary

CLASSES = f'{REPO}/data/datasets/classes.csv'
missing = [str(run_dir) for run_dir, _ in RUNS.values() if not (Path(run_dir) / 'scores.json').is_file()]
if missing:
    raise FileNotFoundError(f'scores.json이 없는 run 폴더가 있습니다: {missing}\n'
                            '[해결] RUNS의 경로를 Drive에서 확인하고, 채점(val 채점 셀)을 마친 run인지 확인하세요.')

for name, (run_dir, variant) in RUNS.items():
    scores = json.loads((Path(run_dir) / 'scores.json').read_text(encoding='utf-8'))
    s = imbalance_summary(scores['per_class'], CLASSES, variant)
    print(f"{name} ({variant}) | val {scores['val']:.4f} (여유 {1 - scores['val']:.4f}) | "
          f"희소 절반 {s['rare_ap']:.4f}  빈번 절반 {s['frequent_ap']:.4f}  gap {s['gap']:.4f}  [{s['n_classes']}개 클래스]")
    for r in class_ap_rows(scores['per_class'], CLASSES, variant)[:N_BOTTOM]:
        print(f"    {r['k_code']}  train {r['n_train']:>5}  val {r['n_val']:>3}  test {r['n_test']:>3}  AP {r['ap']:.4f}  {r['name']}")